# **Laboratorio 8: Ready, Set, Deploy! 👩‍🚀👨‍🚀**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Otoño 2026 </strong></center>

### Cuerpo Docente:

- Profesores: Pablo Badilla, Diego Cortez
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Javier Cruz Araneda
- Nombre de alumno 2: Enzo Toledo Venegas

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/enzo-toledo/MDS7202)

## Temas a tratar

- Entrenamiento y registro de modelos usando MLFlow.
- Despliegue de modelo usando FastAPI
- Containerización del proyecto usando Docker

### Objetivos principales del laboratorio

- Generar una solución a un problema a partir de ML
- Desplegar su solución usando MLFlow, FastAPI y Docker

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# **Introducción**

<p align="center">
  <img src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExODJnMHJzNzlkNmQweXoyY3ltbnZ2ZDlxY2c0aW5jcHNzeDNtOXBsdCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/AbPdhwsMgjMjax5reo/giphy.gif" width="400">
</p>



Consumida en la tristeza el despido de Renacín, Smapina ha decaído en su desempeño, lo que se ha traducido en un irregular tratamiento del agua. Esto ha implicado una baja en la calidad del agua, llegando a haber algunos puntos de la comuna en la que el vital elemento no es apto para el consumo humano. Es por esto que la sanitaria pública de la municipalidad de Maipú se ha contactado con ustedes para que le entreguen una urgente solución a este problema (a la vez que dejan a Smapina, al igual que Renacín, sin trabajo 😔).

El problema que la empresa le ha solicitado resolver es el de elaborar un sistema que les permita saber si el agua es potable o no. Para esto, la sanitaria les ha proveido una base de datos con la lectura de múltiples sensores IOT colocados en diversas cañerías, conductos y estanques. Estos sensores señalan nueve tipos de mediciones químicas y más una etiqueta elaborada en laboratorio que indica si el agua es potable o no el agua.

La idea final es que puedan, en el caso que el agua no sea potable, dar un aviso inmediato para corregir el problema. Tenga en cuenta que parte del equipo docente vive en Maipú y su intoxicación podría implicar graves problemas para el cierre del curso.

Atributos:

1. pH value
2. Hardness
3. Solids (Total dissolved solids - TDS)
4. Chloramines
5. Sulfate
6. Conductivity
7. Organic_carbon
8. Trihalomethanes
9. Turbidity

Variable a predecir:

10. Potability (1 si es potable, 0 no potable)

Descripción de cada atributo se pueden encontrar en el siguiente link: [dataset](https://www.kaggle.com/adityakadiwal/water-potability)

# **1. Optimización de modelos con Optuna + MLFlow (2.0 puntos)**

El objetivo de esta sección es que ustedes puedan combinar Optuna con MLFlow para poder realizar la optimización de los hiperparámetros de sus modelos.

Como aún no hemos hablado nada sobre `MLFlow` cabe preguntarse: **¡¿Qué !"#@ es `MLflow`?!**

<p align="center">
  <img src="https://media.tenor.com/eusgDKT4smQAAAAC/matthew-perry-chandler-bing.gif" width="400">
</p>

## **MLFlow**

`MLflow` es una plataforma de código abierto que simplifica la gestión y seguimiento de proyectos de aprendizaje automático. Con sus herramientas, los desarrolladores pueden organizar, rastrear y comparar experimentos, además de registrar modelos y controlar versiones.

<p align="center">
  <img src="https://spark.apache.org/images/mlflow-logo.png" width="350">
</p>

Si bien esta plataforma cuenta con un gran número de herramientas y funcionalidades, en este laboratorio trabajaremos con dos:
1. **Runs**: Registro que constituye la información guardada tras la ejecución de un entrenamiento. Cada `run` tiene su propio run_id, el cual sirve como identificador para el entrenamiento en sí mismo. Dentro de cada `run` podremos acceder a información como los hiperparámetros utilizados, las métricas obtenidas, las librerías requeridas y hasta nos permite descargar el modelo entrenado.
2. **Experiments**: Se utilizan para agrupar y organizar diferentes ejecuciones de modelos (`runs`). En ese sentido, un experimento puede agrupar 1 o más `runs`. De esta manera, es posible también registrar métricas, parámetros y archivos (artefactos) asociados a cada experimento.

### **Todo bien pero entonces, ¿cómo se usa en la práctica `MLflow`?**

Es sencillo! Considerando un problema de machine learning genérico, podemos registrar la información relevante del entrenamiento ejecutando `mlflow.autolog()` antes entrenar nuestro modelo. Veamos este bonito ejemplo facilitado por los mismos creadores de `MLflow`:

```python
#!pip install mlflow
import mlflow # importar mlflow

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

db = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)

mlflow.autolog() # registrar automáticamente información del entrenamiento
with mlflow.start_run(): # delimita inicio y fin del run
    # aquí comienza el run
    rf.fit(X_train, y_train) # train the model
    predictions = rf.predict(X_test) # Use the model to make predictions on the test dataset.
    # aquí termina el run
```

Si ustedes ejecutan el código anterior en sus máquinas locales (desde un jupyter notebook por ejemplo) se darán cuenta que en su directorio *root* se ha creado la carpeta `mlruns`. Esta carpeta lleva el tracking de todos los entrenamientos ejecutados desde el directorio root (importante: si se cambian de directorio y vuelven a ejecutar el código anterior, se creará otra carpeta y no tendrán acceso al entrenamiento anterior). Para visualizar estos entrenamientos, `MLflow` nos facilita hermosa interfaz visual a la que podemos acceder ejecutando:

```
mlflow ui
```

y luego pinchando en la ruta http://127.0.0.1:5000 que nos retorna la terminal. Veamos en vivo algunas de sus funcionalidades!

<p align="center">
  <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZXVuM3A5MW1heDFpa21qbGlwN2pyc2VoNnZsMmRzODZxdnluemo2bCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/3o84sq21TxDH6PyYms/giphy.gif" width="400">
</p>

Les dejamos también algunos comandos útiles:

- `mlflow.create_experiment("nombre_experimento")`: Les permite crear un nuevo experimento para agrupar entrenamientos
- `mlflow.log_metric("nombre_métrica", métrica)`: Les permite registrar una métrica *custom* bajo el nombre de "nombre_métrica"


In [1]:
!uv add mlflow

Resolved 176 packages in 16ms
Checked 167 packages in 265ms


Si tiene problemas puede necesitar ejecutar `uv add "setuptools<82.0.0"`

In [2]:
!uv add "setuptools<82.0.0"

# daba otro error, con esto se arregla.
!uv pip install "protobuf<4.0.0"

Resolved 176 packages in 0.94ms
Checked 167 packages in 7ms
Using Python 3.14.3 environment at: C:\Users\Javier\Desktop\MDS7202\.venv
Resolved 1 package in 83ms
Uninstalled 1 package in 16ms
Installed 1 package in 31ms
 - protobuf==7.35.0
 + protobuf==3.20.3


In [24]:
"""
import mlflow  # importar mlflow
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

db = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)

mlflow.autolog()  # registrar automáticamente información del entrenamiento
with mlflow.start_run():  # delimita inicio y fin del run
    # aquí comienza el run
    rf.fit(X_train, y_train)  # train the model
    predictions = rf.predict(X_test)  # Use the model to make predictions on the test dataset.
    # aquí termina el run
"""

'\nimport mlflow  # importar mlflow\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.datasets import load_diabetes\nfrom sklearn.ensemble import RandomForestRegressor\n\ndb = load_diabetes()\nX_train, X_test, y_train, y_test = train_test_split(db.data, db.target)\n\n# Create and train models.\nrf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)\n\nmlflow.autolog()  # registrar automáticamente información del entrenamiento\nwith mlflow.start_run():  # delimita inicio y fin del run\n    # aquí comienza el run\n    rf.fit(X_train, y_train)  # train the model\n    predictions = rf.predict(X_test)  # Use the model to make predictions on the test dataset.\n    # aquí termina el run\n'

In [25]:
"""
run = mlflow.last_active_run()
info = mlflow.get_run(run.info.run_id)
print(info.data.params)
print(info.data.metrics)
"""

'\nrun = mlflow.last_active_run()\ninfo = mlflow.get_run(run.info.run_id)\nprint(info.data.params)\nprint(info.data.metrics)\n'

In [3]:
# Librerias a utilizar:
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
# Cargar los datos:
df = pd.read_csv("water_potability.csv")
df.head()

df.isnull().sum()

ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64

In [5]:
# Imputamos por la mediana (robusto ante outliers):
df.fillna(df.median(), inplace=True)  # en todo el lab trabajamos así por eso inplace.

# Preparar datos:
X = df.drop("Potability", axis=1)
y = df["Potability"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
df.isnull().sum()

ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64

## **1.1 Combinando Optuna + MLflow (2.0 puntos)**

Ahora que tenemos conocimiento de ambas herramientas, intentemos ahora combinarlas para **más sabor**. El objetivo de este apartado es simple: automatizar la optimización de los parámetros de nuestros modelos usando `Optuna` y registrando de forma automática cada resultado en `MLFlow`.

Considerando el objetivo planteado, se le pide completar la función `optimize_model`, la cual debe:
- **Optimizar los hiperparámetros del modelo `XGBoost` usando `Optuna`.** Realice una cantidad de iteraciones para evitar tiempos de ejecución excesivos (al menos 10)
- **Registrar cada entrenamiento en un experimento nuevo**, asegurándose de que la métrica `f1-score` se registre como `"valid_f1"`. No se deben guardar todos los experimentos en *Default*; en su lugar, cada `experiment` y `run` deben tener nombres interpretables, reconocibles y diferentes a los nombres por defecto (por ejemplo, para un run: "XGBoost con lr 0.1").
- **Devolver el mejor modelo** usando la función `get_best_model` y serializarlo en el disco con `pickle.dump`. Luego, guardar el modelo en la carpeta `/models`.
- **Guardar el código en `optimize.py`**. La ejecución de `python optimize.py` debería ejecutar la función `optimize_model`.
- **Guardar las versiones de las librerías utilizadas** en el desarrollo.

*Hint: Le puede ser útil revisar los parámetros que recibe `mlflow.start_run`*

```python
def get_best_model(experiment_id):
    runs = mlflow.search_runs(experiment_id)
    best_model_id = runs.sort_values("metrics.valid_f1")["run_id"].iloc[0]
    best_model = mlflow.sklearn.load_model("runs:/" + best_model_id + "/model")

    return best_model
```

**Para crear el `optimize.py` se consideró la celda de abajo resumiendo todo lo necesario para crear el .py con lo pedido:**

In [7]:
%%writefile optimize.py
import optuna
import mlflow
import pickle
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

# Repetimos todo para que funcione el .py bien:
df = pd.read_csv("water_potability.csv")
df.fillna(df.median(), inplace=True)

X = df.drop("Potability", axis=1)
y = df["Potability"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


def get_best_model(experiment_id):
    runs = mlflow.search_runs(experiment_id)
    best_model_id = runs.sort_values("metrics.valid_f1")["run_id"].iloc[0]
    best_model = mlflow.sklearn.load_model("runs:/" + best_model_id + "/model")
    return best_model


def optimize_model():
    # Crear experimento:
    experiment_name = "XGBoost_Potabilidad_Agua"

    # Evitar error por si existe:
    existing = mlflow.get_experiment_by_name(experiment_name)
    if existing is None:
        experiment_id = mlflow.create_experiment(experiment_name)
    else:
        experiment_id = existing.experiment_id

    # Definir función para trials con Optuna (como en lab_7 pero con XGBoost y MLflow)
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        }

        # Nombre de run interpretable, usando el learning_rate como referencia
        run_name = f"XGBoost con lr {round(params['learning_rate'], 3)}"

        # Cada trial es un run dentro del experimento:
        with mlflow.start_run(experiment_id=experiment_id, run_name=run_name):
            model = XGBClassifier(**params, eval_metric="logloss", random_state=42)
            model.fit(X_train, y_train)

            # Predecir sobre test y calcular f1
            preds = model.predict(X_test)
            valid_f1 = f1_score(y_test, preds, average="macro")  # mejor ante desbalanceo el macro.

            # Registrar manualmente parametros, metrica y modelo
            mlflow.log_params(params)
            mlflow.log_metric("valid_f1", valid_f1)
            mlflow.sklearn.log_model(model, "model")

        return valid_f1

    # Crear el estudio y optimizar (maximizar f1), al menos 10 iteraciones
    study = optuna.create_study(
        direction="maximize", study_name="XGBoost-Potabilidad", sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(objective, n_trials=15)

    # Recuperar el mejor modelo registrado en MLflow
    best_model = get_best_model(experiment_id)

    # 5. Guardar en models/ el mejor modelo obtenido:
    os.makedirs("models", exist_ok=True)
    with open("models/best_model.pkl", "wb") as f:
        pickle.dump(best_model, f)

    print("Mejor f1 obtenido:", study.best_value)
    print("Mejores hiperparametros:", study.best_params)

    # Guardar versiones de librerías:
    #!uv pip freeze > requirements.txt mover.

    return best_model


if __name__ == "__main__":
    optimize_model()

Overwriting optimize.py


In [ ]:
#!uv pip freeze --no-color > requirements.txt Se crea despues el requierements.txt adecuado para el docker para usarlo más limpio.

In [8]:
# Para ejecutar el script directamente desde el notebook: (en testeos tarda 1 minuto aprox.)
!python optimize.py

Mejor f1 obtenido: 0.6017586127104686
Mejores hiperparametros: {'n_estimators': 96, 'max_depth': 4, 'learning_rate': 0.16217936517334897, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021}


c:\Users\Javier\Desktop\MDS7202\.venv\Lib\site-packages\mlflow\utils\autologging_utils\versioning.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
[I 2026-06-09 16:41:05,132] A new study created in memory with name: XGBoost-Potabilidad
2026/06/09 16:41:10 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Javier\AppData\Local\Temp\tmpelw1267j\model\model.pkl, flavor: sklearn), fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback.
2026/06/09 16:41:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
[I 2026-

**Observación:** Resultados sin imputar datos (dejando los NaN):
> Mejor f1 obtenido: 0.5937161903440265
> Mejores hiperparametros: {'n_estimators': 230, 'max_depth': 7, 'learning_rate': 0.1975195386194739, 'subsample': 0.8674501912332678, 'colsample_bytree': 0.796896148820497}

Resultados imputando por la mediana:
> Mejor f1 obtenido: 0.6017586127104686
> Mejores hiperparametros: {'n_estimators': 96, 'max_depth': 4, 'learning_rate': 0.16217936517334897, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021}

La mejora con imputación es prácticamente nula (~0.008 F1) porque XGBoost tiene manejo de NaN.

> **Importante:** Se considera que se ejecuta `python optimize.py` desde la carpeta del lab_8 y no la raíz del repo. Además, antes de seguir con la otra sección, hay que correr el código para que se generen mlruns y models con todo lo anterior y por eso se dejó antes la celda ya para ejecutarlo directo.

# **2. FastAPI (2.0 puntos)**

<div align="center">
  <img src="https://media3.giphy.com/media/YQitE4YNQNahy/giphy-downsized-large.gif" width="500">
</div>

Con el modelo ya entrenado, la idea de esta sección es generar una API REST a la cual se le pueda hacer *requests* para así interactuar con su modelo. En particular, se le pide:

- Guardar el código de esta sección en el archivo `main.py`. Note que ejecutar `python main.py` debería levantar el servidor en el puerto por defecto.
- Defina `GET` con ruta tipo *home* que describa brevemente su modelo, el problema que intenta resolver, su entrada y salida.
- Defina un `POST` a la ruta `/potabilidad/` donde utilice su mejor optimizado para predecir si una medición de agua es o no potable. Por ejemplo, una llamada de esta ruta con un *body*:

```json
{
   "ph":10.316400384553162,
   "Hardness":217.2668424334475,
   "Solids":10676.508475429378,
   "Chloramines":3.445514571005745,
   "Sulfate":397.7549459751925,
   "Conductivity":492.20647361771086,
   "Organic_carbon":12.812732207582542,
   "Trihalomethanes":72.28192021570328,
   "Turbidity":3.4073494284238364
}
```

Su servidor debería retornar una respuesta HTML con código 200 con:


```json
{
  "potabilidad": 0 # respuesta puede variar según el clasificador que entrenen
}
```

**`HINT:` Recuerde que puede utilizar [http://localhost:8000/docs](http://localhost:8000/docs) para hacer un `POST`.**

In [9]:
!uv add fastapi uvicorn

Resolved 176 packages in 0.90ms
Uninstalled 1 package in 7ms
Installed 1 package in 30ms
 - protobuf==3.20.3
 + protobuf==7.35.0


In [10]:
%%writefile main.py
import pickle
import pandas as pd
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

# Cargar el mejor modelo de la parte anterior:
with open("models/best_model.pkl", "rb") as f:
    model = pickle.load(f)

app = FastAPI(title="FastAPI Potabilidad del Agua")


# Esquema del body (mismas columnas y orden que el dataset de entrenamiento!):
class WaterSample(BaseModel):
    ph: float
    Hardness: float
    Solids: float
    Chloramines: float
    Sulfate: float
    Conductivity: float
    Organic_carbon: float
    Trihalomethanes: float
    Turbidity: float


@app.get("/")
def home():
    return {
        "modelo": "XGBoost optimizado con Optuna + MLflow",
        "problema": "Clasificacion binaria de potabilidad del agua a partir de 9 mediciones.",
        "entrada": "JSON con: ph, Hardness, Solids, Chloramines, Sulfate, "
        "Conductivity, Organic_carbon, Trihalomethanes, Turbidity",
        "salida": "JSON {'potabilidad': 0 o 1}, donde 1 = agua potable y 0 = no potable.",
    }


@app.post("/potabilidad/")
def predict_potabilidad(sample: WaterSample):
    # Convertir el body a DataFrame respetando los nombres de columnas del entrenamiento:
    X = pd.DataFrame([sample.model_dump()])
    pred = model.predict(X)
    return {"potabilidad": int(pred[0])}


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

Overwriting main.py


In [35]:
# Ejecutar el servidor: (NO EJECUTAR AQUÍ)
#!python main.py

> NO ejecutar el servidor desde el notebook ya que una celda con "!python main.py" bloquea el kernel y, al detenerla, el proceso queda ocupando el puerto 8000 (error 10048 testeado antes que ocurría). Hacerlo sí o sí desde una terminal en esta ubicación del lab_8 y así funciona bien y se puede ejecutar el request de la celda de más abajo bien.


**`HINT:` Recuerde que puede utilizar [http://localhost:8000/docs](http://localhost:8000/docs) para hacer un `POST`.**

**Aquí testeamos que funcione una request como la del ejemplo:**

> Mientres esté corriendo el server ejecutándolo desde la terminal la request de abajo funcionará bien (entrega el resultado del modelo y el código 200), y cuando se cierre el server desde la terminal (ctrl+C) ejecutar la request de abajo debería dar error.

In [11]:
import requests

body = {
    "ph": 10.316400384553162,
    "Hardness": 217.2668424334475,
    "Solids": 10676.508475429378,
    "Chloramines": 3.445514571005745,
    "Sulfate": 397.7549459751925,
    "Conductivity": 492.20647361771086,
    "Organic_carbon": 12.812732207582542,
    "Trihalomethanes": 72.28192021570328,
    "Turbidity": 3.4073494284238364,
}

r = requests.post("http://localhost:8000/potabilidad/", json=body)
print(r.status_code)
print(r.json())

200
{'potabilidad': 0}


# **3. Docker (2 puntos)**

<div align="center">
  <img src="https://miro.medium.com/v2/resize:fit:1400/1*9rafh2W0rbRJIKJzqYc8yA.gif" width="500">
</div>

Tras el éxito de su aplicación web para generar la salida, Smapina le solicita que genere un contenedor para poder ejecutarla en cualquier computador de la empresa de agua potable.

## **3.1 Creación de Container (1 punto)**

Cree un Dockerfile que use una imagen base de Python, copie los archivos del proyecto e instale las dependencias desde un `requirements.txt`. Con esto, construya y ejecute el contenedor Docker para la API configurada anteriormente. Entregue el código fuente (incluyendo `main.py`, `requirements.txt`, y `Dockerfile`) y la imagen Docker de la aplicación. Para la dockerización, asegúrese de cumplir con los siguientes puntos:

1. **Generar un archivo `.dockerignore`** que ignore carpetas y archivos innecesarios dentro del contenedor.
2. **Configurar un volumen** que permita la persistencia de los datos en una ruta local del computador.
3. **Exponer el puerto** para acceder a la ruta de la API sin tener que entrar al contenedor directamente.
4. **Incluir imágenes en el notebook** que muestren la ejecución del contenedor y los resultados obtenidos.
5. **Revisar y comentar los recursos utilizados por el contenedor**. Analice si los contenedores son livianos en términos de recursos.

In [14]:
%%writefile requirements.txt
fastapi
uvicorn
pandas
scikit-learn
xgboost

Overwriting requirements.txt


**Respuestas:**

Se adjuntan capturas de pantalla que evidencian el buen funcionamiento del container. 

Cabe destacar que se utilizó un archivo `requirements.txt` separado con las dependencias mínimas necesarias para ejecutar la API (fastapi, uvicorn, pandas, scikit-learn, xgboost), en lugar del completo que puede ser generado con pip freeze por el entorno de desarrollo. Esto se debe a que el último incluye librerías propias del entorno Jupyter (como ipython e ipykernel) con versiones fijadas para Python 3.14, las cuales no son compatibles con la imagen base python:3.11-slim utilizada en el contenedor.

* Docker corriendo:

![Docker corriendo](img/docker_corriendo.png)

* API respondiendo:

![API respondiendo](img/api_respondiendo.png)

* Recursos ocupados por el Docker:

![Recursos Docker](img/recursos_docker.png)

El contenedor potabilidad-container presenta un consumo de recursos reducido ya que utiliza apenas `120 MiB ` de RAM sobre un límite de `~15 GiB` disponibles y estando en reposo, presenta un uso de CPU de $0.08 \%$. El tráfico de red es mínimo (`4.04 kB` / `4.53 kB`), lo que es consistente con una API que no ha recibido muchas solicitudes. Estos valores confirman que los contenedores Docker son soluciones ligeras en términos de recursos, ya que encapsulan únicamente lo necesario para ejecutar la aplicación, a diferencia de una máquina virtual que requeriría reservar recursos para un sistema operativo completo.

## **3.2 Preguntas de Smapina (1 punto)**
Tras haber experimentado con Docker, Smapina desea profundizar más en el tema y decide realizarle las siguientes consultas:

- ¿Cómo se diferencia Docker de una máquina virtual (VM)?
- ¿Cuál es la diferencia entre usar Docker y ejecutar la aplicación directamente en el sistema local?
- ¿Cómo asegura Docker la consistencia entre diferentes entornos de desarrollo y producción?
- ¿Cómo se gestionan los volúmenes en Docker para la persistencia de datos?
- ¿Qué son Dockerfile y docker-compose.yml, y cuál es su propósito?

**Respuestas:**

* Docker utiliza contenedores que comparten el kernel del sistema operativo anfitrión, lo que lo hace significativamente más liviano y rápido de iniciar que una máquina virtual. Una VM, en cambio, virtualiza hardware completo e incluye su propio sistema operativo, lo que implica un mayor consumo de recursos (RAM, CPU y almacenamiento).

* Ejecutar la aplicación directamente en el sistema local depende del entorno específico de la máquina, es decir, depende de la versión de Python que se ocupe, las librerías instaladas y la configuración del sistema operativo. Esto puede provocar problemas al intentar correr aplicaciones en equipos diferentes al usado en un inicio para su desarrollo. Docker elimina esta dependencia al empaquetar la aplicación junto con todas sus dependencias en un contenedor aislado, garantizando que se ejecute de forma idéntica en cualquier máquina que tenga Docker instalado.

* Docker garantiza la consistencia a través de la imagen, que es inmutable y contiene exactamente las mismas dependencias, versión de Python y configuración independientemente de dónde se ejecute. Al construir la imagen a partir de un Dockerfile versionado en el repositorio, cualquier miembro del equipo o servidor de producción puede reproducir el mismo entorno con un simple `docker build`, eliminando discrepancias entre entornos.

* Por defecto, los datos generados dentro de un contenedor se pierden al detenerlo, ya que el sistema de archivos del contenedor es efímero. Los volúmenes permiten montar una carpeta del sistema anfitrión dentro del contenedor, de modo que los datos escritos en esa ruta persisten más allá del ciclo de vida del contenedor. En este proyecto se configuró un volumen que enlaza la carpeta `data/` del proyecto local con la ruta `/app/data` dentro del contenedor, de modo que cualquier archivo generado por la aplicación queda guardado en el disco del equipo anfitrión y no se pierde al detener el contenedor.

* El Dockerfile es un archivo de texto que define paso a paso cómo construir la imagen Docker de una aplicación: qué imagen base usar, qué dependencias instalar, qué archivos copiar y cómo iniciar el servicio. Por su parte, docker-compose.yml es un archivo de configuración que permite definir y orquestar múltiples contenedores como un único servicio, especificando sus dependencias, redes, volúmenes y variables de entorno. Mientras el Dockerfile construye una imagen individual, docker-compose coordina el despliegue de sistemas compuestos por varios servicios, como una API junto a una base de datos.

# Conclusión

Éxito!
<div align="center">
  <img src="https://i.pinimg.com/originals/55/f5/fd/55f5fdc9455989f8caf7fca7f93bd96a.gif" width="500">
</div>